# PaySim (UPI) — XGBoost Fraud Detection Training

**Fully self-contained** — no external imports needed. Just add the PaySim dataset and run.

**Dataset:** [Synthetic Financial Datasets for Fraud Detection](https://www.kaggle.com/datasets/ealaxi/paysim1)  
**Output:** `artifacts/` folder downloadable from the Output tab as a zip.

## 1. Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn

## 2. Configuration

In [ ]:
# Set to None for full training, or a small number (for example 10000) for dry-run
SAMPLE_ROWS = 10000

ARTIFACT_VERSION = "v1"
N_ESTIMATORS = 300
MAX_DEPTH = 6
LEARNING_RATE = 0.1
TRAIN_RATIO = 0.8

# Kaggle dataset path candidates — first existing path will be used
DATA_PATH_CANDIDATES = [
    "/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv",
    "/kaggle/input/archive-12/PS_20174392719_1491204439457_log.csv",
    "/kaggle/input/archive-12/archive (12)/PS_20174392719_1491204439457_log.csv",
]
OUTPUT_DIR = "/kaggle/working/artifacts"

## 3. Load & Preview Data

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find PaySim CSV. Update DATA_PATH_CANDIDATES for your Kaggle mount."
    )

read_kwargs = {"nrows": SAMPLE_ROWS} if SAMPLE_ROWS is not None else {}
if SAMPLE_ROWS is not None:
    print(f"Running dry-run with first {SAMPLE_ROWS} rows")
df = pd.read_csv(DATA_PATH, **read_kwargs)

print(f"Loaded data from: {DATA_PATH}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Fraud rate: {df['isFraud'].mean():.4f}")
df.head()

## 4. Feature Engineering

9 causal features — history-dependent features only use prior rows (no future leakage).

In [ ]:
EPSILON = 1e-6
RISKY_TYPES = {"TRANSFER", "CASH_OUT"}
STEPS_24H = 24
STEPS_7D = 168

PAYSIM_TRAINING_FEATURES = [
    "type_risk_flag",
    "amount_to_orig_balance_ratio",
    "orig_balance_consistency_error",
    "dest_balance_consistency_error",
    "orig_balance_drain_pct",
    "sender_txn_count_24h",
    "sender_amount_zscore_7d",
    "sender_dest_pair_novelty",
    "dest_inbound_txn_count_24h",
]

PAYSIM_ONLINE_FEATURES = [
    "type_risk_flag",
    "amount_to_orig_balance_ratio",
    "orig_balance_consistency_error",
    "sender_txn_count_24h",
    "sender_amount_zscore_7d",
    "sender_dest_pair_novelty",
]

def build_paysim_features(df):
    """Build all 9 PaySim training features causally."""
    df = df.sort_values("step").reset_index(drop=True)

    # --- Vectorized features ---
    df["type_risk_flag"] = df["type"].isin(RISKY_TYPES).astype(int)

    df["amount_to_orig_balance_ratio"] = np.clip(
        df["amount"] / np.maximum(df["oldbalanceOrg"], EPSILON), 0.0, 1.0
    )

    df["orig_balance_consistency_error"] = np.clip(
        np.abs(df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"])
        / np.maximum(df["oldbalanceOrg"], EPSILON), 0.0, 1.0
    )

    # Merchant destinations (M-prefix) → neutral 0.0
    is_merchant = df["nameDest"].str.startswith("M")
    raw_dest_err = np.clip(
        np.abs(df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"])
        / np.maximum(df["amount"], EPSILON), 0.0, 1.0
    )
    df["dest_balance_consistency_error"] = np.where(is_merchant, 0.0, raw_dest_err)

    # TRANSFER/CASH_OUT only; 0.0 otherwise
    is_risky = df["type"].isin(RISKY_TYPES)
    raw_drain = np.clip(
        (df["oldbalanceOrg"] - df["newbalanceOrig"])
        / np.maximum(df["oldbalanceOrg"], EPSILON), 0.0, 1.0
    )
    df["orig_balance_drain_pct"] = np.where(is_risky, raw_drain, 0.0)

    # --- History-dependent features (iterative, causal) ---
    n = len(df)
    sender_txn_count_24h = np.zeros(n, dtype=np.int64)
    sender_amount_zscore_7d = np.zeros(n, dtype=np.float64)
    sender_dest_pair_novelty = np.zeros(n, dtype=np.int64)
    dest_inbound_txn_count_24h = np.zeros(n, dtype=np.int64)

    sender_history = {}   # sender → [(step, amount), ...]
    sender_dest_pairs = {}  # sender → set of dests
    dest_history = {}     # dest → [step, ...]

    for i in range(n):
        row = df.iloc[i]
        sender = row["nameOrig"]
        dest = row["nameDest"]
        step = int(row["step"])
        amount = float(row["amount"])

        hist = sender_history.get(sender, [])
        sender_txn_count_24h[i] = sum(1 for s, _ in hist if step - s <= STEPS_24H)

        amounts_7d = [a for s, a in hist if step - s <= STEPS_7D]
        if len(amounts_7d) >= 2:
            mean_v = np.mean(amounts_7d)
            std_v = np.std(amounts_7d)
            if std_v > EPSILON:
                sender_amount_zscore_7d[i] = (amount - mean_v) / std_v

        seen = sender_dest_pairs.get(sender, set())
        sender_dest_pair_novelty[i] = 0 if dest in seen else 1

        d_hist = dest_history.get(dest, [])
        dest_inbound_txn_count_24h[i] = sum(1 for s in d_hist if step - s <= STEPS_24H)

        # Update AFTER scoring (causal)
        sender_history.setdefault(sender, []).append((step, amount))
        sender_dest_pairs.setdefault(sender, set()).add(dest)
        dest_history.setdefault(dest, []).append(step)

        if i % 500000 == 0 and i > 0:
            print(f"  Feature progress: {i}/{n} rows ({i*100//n}%)")

    df["sender_txn_count_24h"] = sender_txn_count_24h
    df["sender_amount_zscore_7d"] = sender_amount_zscore_7d
    df["sender_dest_pair_novelty"] = sender_dest_pair_novelty
    df["dest_inbound_txn_count_24h"] = dest_inbound_txn_count_24h

    output_cols = PAYSIM_TRAINING_FEATURES + ["isFraud"]
    return df[output_cols].copy()

print("Building features (this may take a few minutes on full data)...")
features_df = build_paysim_features(df)
print(f"Done! Shape: {features_df.shape}")
features_df.describe()

## 5. Train / Validation Split

In [ ]:
feature_cols = PAYSIM_TRAINING_FEATURES
TARGET = "isFraud"

split_idx = int(len(features_df) * TRAIN_RATIO)
train_df = features_df.iloc[:split_idx]
val_df = features_df.iloc[split_idx:]

X_train = train_df[feature_cols].values
y_train = train_df[TARGET].values.astype(int)
X_val = val_df[feature_cols].values
y_val = val_df[TARGET].values.astype(int)

n_neg = int(np.sum(y_train == 0))
n_pos = max(int(np.sum(y_train == 1)), 1)
scale_pos_weight = n_neg / n_pos

print(f"Train: {len(train_df)} rows ({np.mean(y_train):.4f} fraud rate)")
print(f"Val:   {len(val_df)} rows ({np.mean(y_val):.4f} fraud rate)")
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

## 6. Train XGBoost

In [ ]:
import xgboost as xgb

# Prefer GPU on Kaggle when available; otherwise fall back to CPU
try:
    probe_X = np.array([[0.0, 0.0], [1.0, 1.0]])
    probe_y = np.array([0, 1])
    _t = xgb.XGBClassifier(
        device="cuda",
        n_estimators=1,
        max_depth=1,
        tree_method="hist",
        eval_metric="aucpr",
        random_state=42,
    )
    _t.fit(probe_X, probe_y, verbose=False)
    device = "cuda"
except Exception as exc:
    device = "cpu"
    print(f"GPU unavailable, falling back to CPU: {exc}")
print(f"Using device: {device}")

model = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    scale_pos_weight=scale_pos_weight,
    device=device,
    tree_method="hist",
    eval_metric="aucpr",
    use_label_encoder=False,
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=10)

## 7. Evaluate

In [ ]:
from sklearn.metrics import (
    average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score,
    precision_recall_curve,
)

y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

pr_auc = average_precision_score(y_val, y_pred_proba)
prec = precision_score(y_val, y_pred, zero_division=0)
rec = recall_score(y_val, y_pred, zero_division=0)
f1 = f1_score(y_val, y_pred, zero_division=0)
cm = confusion_matrix(y_val, y_pred)

metrics = {
    "threshold": 0.5,
    "precision": round(float(prec), 4),
    "recall": round(float(rec), 4),
    "f1": round(float(f1), 4),
    "pr_auc": round(float(pr_auc), 4),
    "confusion_matrix": cm.tolist(),
}

print("=" * 50)
print("  Classification Report")
print("=" * 50)
print(f"  Precision:  {metrics['precision']}")
print(f"  Recall:     {metrics['recall']}")
print(f"  F1 Score:   {metrics['f1']}")
print(f"  PR-AUC:     {metrics['pr_auc']}")
print(f"\n  Confusion Matrix:")
print(f"    TN={cm[0][0]:>6}  FP={cm[0][1]:>6}")
print(f"    FN={cm[1][0]:>6}  TP={cm[1][1]:>6}")
print("=" * 50)

## 8. Suggest Thresholds

In [ ]:
prec_vals, rec_vals, pr_thresholds = precision_recall_curve(y_val, y_pred_proba)

def find_threshold(target_prec):
    for i, p in enumerate(prec_vals[:-1]):
        if p >= target_prec and i < len(pr_thresholds):
            return float(pr_thresholds[i])
    return float(np.median(pr_thresholds)) if len(pr_thresholds) > 0 else 0.5

allow_to_review = float(np.clip(find_threshold(0.3), 0.2, 0.6))
review_to_block = float(np.clip(find_threshold(0.8), 0.7, 0.95))

thresholds = {
    "level": {
        "low_to_medium": round(allow_to_review, 3),
        "medium_to_high": round(review_to_block - 0.15, 3),
    },
    "decision": {
        "allow_to_review": round(allow_to_review, 3),
        "review_to_block": round(review_to_block, 3),
    },
}

print(f"Level thresholds:    {thresholds['level']}")
print(f"Decision thresholds: {thresholds['decision']}")

## 9. Export Artifacts

Writes all files to the output directory. Download the `artifacts/` folder from the Kaggle Output tab as a zip.

In [ ]:
import json
import joblib
import shutil
from pathlib import Path
from datetime import datetime, timezone

# Build manifest
manifest = {
    "domain": "paysim",
    "artifact_version": ARTIFACT_VERSION,
    "model_family": "xgboost",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "feature_order": PAYSIM_TRAINING_FEATURES,
    "required_raw_fields": ["step", "type", "amount", "nameOrig", "oldbalanceOrg",
                            "newbalanceOrig", "nameDest", "oldbalanceDest", "newbalanceDest"],
    "online_features": PAYSIM_ONLINE_FEATURES,
    "training_features": PAYSIM_TRAINING_FEATURES,
    "score_mapping": {
        "heuristic_score": "scores.heuristic",
        "supervised_probability": "scores.supervised",
        "final_risk_score": "risk.score",
    },
    "alert_thresholds": {
        "low_to_medium": thresholds["level"]["low_to_medium"],
        "medium_to_high": thresholds["level"]["medium_to_high"],
        "allow_to_review": thresholds["decision"]["allow_to_review"],
        "review_to_block": thresholds["decision"]["review_to_block"],
    },
}

metadata = {
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "data_source": DATA_PATH,
    "sample_rows": SAMPLE_ROWS,
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "device": device,
    "n_estimators": N_ESTIMATORS,
    "max_depth": MAX_DEPTH,
    "learning_rate": LEARNING_RATE,
    "scale_pos_weight": round(scale_pos_weight, 2),
}

feature_defaults = {f: 0.0 for f in feature_cols}

# Write all files
version_dir = Path(OUTPUT_DIR) / "paysim" / ARTIFACT_VERSION
version_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, version_dir / "supervised_model.joblib")

for name, data in [
    ("manifest.json", manifest),
    ("feature_order.json", feature_cols),
    ("feature_defaults.json", feature_defaults),
    ("thresholds.json", thresholds),
    ("metrics.json", metrics),
    ("metadata.json", metadata),
]:
    (version_dir / name).write_text(json.dumps(data, indent=2, default=str) + "\n")

# Write latest manifest for backend
manifests_dir = Path(OUTPUT_DIR) / "manifests"
manifests_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(version_dir / "manifest.json", manifests_dir / "paysim_latest.json")

print(f"\n✅ Artifacts exported to: {version_dir}")
print(f"✅ Latest manifest: {manifests_dir / 'paysim_latest.json'}")
print(f"\n📦 Download the 'artifacts/' folder from the Output tab.")

## 10. Verify Output

In [ ]:
import os

print("Exported files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, file)) / 1024
        print(f"{subindent}{file} ({size_kb:.1f} KB)")